In [ ]:
import asyncio
import itertools
from typing import List
from benchmarks import benchmark_orchestrator
from benchmarks.answer_generators import (
    AdkAnswerGenerator,
    GeminiAnswerGenerator,
    GroundTruthAnswerGenerator,
    TrivialAnswerGenerator,
)
from benchmarks.data_models import BenchmarkRunResult
import pandas as pd
from pathlib import Path
from benchmarks.logger import JsonTraceLogger
import re


# Set pandas display options
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

# ANSI escape codes for colors
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

def permute(cls, **kwargs):
    """Helper to generate permutations of class instances."""
    keys = kwargs.keys()
    values = kwargs.values()
    for instance_values in itertools.product(*values):
        yield cls(**dict(zip(keys, instance_values)))

# Read context from llms.txt
# try:
#     with open("llms.txt", "r", encoding="utf-8") as f:
#         llms_context = f.read()
# except FileNotFoundError:
#     print(f"{bcolors.WARNING}Warning: llms.txt not found. Proceeding without context.{bcolors.ENDC}")
#     llms_context = ""

# # Read context from llms-relevant.txt
# try:
#     with open("llms-relevant.txt", "r", encoding="utf-8") as f:
#         llms_context_relevant = f.read()
# except FileNotFoundError:
#     print(f"{bcolors.WARNING}Warning: llms.txt not found. Proceeding without context.{bcolors.ENDC}")
#     llms_context_relevant = ""    

logger = JsonTraceLogger(output_dir="traces")

async def run_comparison() -> List[BenchmarkRunResult]:
    """Sets up and runs the benchmark comparison."""
    print("Configuring benchmark run...")
    
    benchmark_suites = [
        "benchmarks/benchmark_definitions/api_understanding/benchmark.yaml",
        "benchmarks/benchmark_definitions/fix_errors/benchmark.yaml",
        "benchmarks/benchmark_definitions/diagnose_setup_errors_mc/benchmark.yaml",
        "benchmarks/benchmark_definitions/configure_adk_features_mc/benchmark.yaml",
        "benchmarks/benchmark_definitions/predict_runtime_behavior_mc/benchmark.yaml",
    ]
    
    answer_generators = [
        GroundTruthAnswerGenerator(),
        TrivialAnswerGenerator(),
        *permute(
            GeminiAnswerGenerator,
            model_name=["gemini-2.5-flash"],
            context=[None, Path("llms.txt"), Path("llms-relevant.txt")],
        ),
        # AdkAnswerGenerator(),
    ]
    
    print("Executing benchmarks...")
    results = await benchmark_orchestrator.run_benchmarks(
        benchmark_suites=benchmark_suites, 
        answer_generators=answer_generators,
        max_concurrency=10,
        logger=logger,
    )
    
    return results

def extract_error_type(row) -> str:
    """Extracts error type from the result row."""
    if "error_type" in row and pd.notna(row["error_type"]):
        # If it's an Enum object (from pydantic validation), get its value
        et = row["error_type"]
        if hasattr(et, "value"):
            return et.value
        return str(et)
    return "OtherError"

def analyze_logs(
    results_df: pd.DataFrame, generator_name: str, result_type: str = 'fail'
) -> None:
    """Filters and displays benchmark results for a specific generator and result type."""
    
    result_value = 1 if result_type.lower() == 'pass' else 0
    
    print(f"{bcolors.HEADER}--- Analyzing {result_type.upper()}S for {generator_name} ---{bcolors.ENDC}")
    
    filtered_df = results_df[
        (results_df['answer_generator'] == generator_name) & 
        (results_df['result'] == result_value)
    ]
    
    if filtered_df.empty:
        print(f"{bcolors.OKGREEN}No {result_type}s found for {generator_name}.{bcolors.ENDC}")
        return
    
    for _, row in filtered_df.iterrows():
        print(f"{bcolors.WARNING}Suite: {row['suite']}{bcolors.ENDC}")
        print(f"{bcolors.WARNING}Benchmark: {row['benchmark_name']}{bcolors.ENDC}")
        print(f"{bcolors.OKCYAN}  Answer:{bcolors.ENDC}\n    {row['answer']}")
        if result_type.lower() == 'fail':
            print(f"{bcolors.FAIL}  Validation Error:{bcolors.ENDC}\n    {row['validation_error']}")
            if "temp_test_file" in row and pd.notna(row["temp_test_file"]):
                print(f"{bcolors.OKBLUE}  Temp File:{bcolors.ENDC} {row['temp_test_file']}")
        print("-" * 40)


JSON trace log will be written to traces/trace_2025-11-25_22-57-23.jsonl


In [2]:
# Execute the benchmarks
results = await run_comparison()
raw_results_df = pd.DataFrame([r.model_dump() for r in results])

Configuring benchmark run...
Executing benchmarks...
--- Loading benchmark suite: benchmarks/benchmark_definitions/api_understanding/benchmark.yaml ---
  - Queuing tests for answer generator: GroundTruthAnswerGenerator
  - Queuing tests for answer generator: TrivialAnswerGenerator
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)-with-context-llms.txt
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)-with-context-llms-relevant.txt
--- Loading benchmark suite: benchmarks/benchmark_definitions/fix_errors/benchmark.yaml ---
  - Queuing tests for answer generator: GroundTruthAnswerGenerator
  - Queuing tests for answer generator: TrivialAnswerGenerator
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)-with-context-llms.txt
  - Queuing tests for

100%|██████████| 985/985 [01:11<00:00, 13.83it/s]

JSON trace log written to traces/trace_2025-11-25_22-57-23.jsonl


In [3]:
raw_results_df.columns

Index(['suite', 'benchmark_name', 'answer_generator', 'status', 'result',
       'answer', 'rationale', 'validation_error', 'error_type',
       'temp_test_file', 'latency'],
      dtype='object')

In [4]:
if not raw_results_df.empty:
    raw_results_df["suite"] = raw_results_df["suite"].apply(lambda x: x.split("/")[-2])
    raw_results_df["final_error_type"] = raw_results_df.apply(extract_error_type, axis=1)

In [5]:
if not raw_results_df.empty:
    # 1. General Pass/Total Summary
    summary_df = (
        raw_results_df.groupby(["answer_generator", "suite"])
        .agg(
            passed=("result", "sum"),
            total=("result", "count"),
        )
    )
    summary_df["pass_rate"] = summary_df["passed"] / summary_df["total"]

    print(f"{bcolors.HEADER}--- Benchmark Summary ---{bcolors.ENDC}")
    print(summary_df)
    print("\n")

    # 2. Detailed Error Breakdown with Ratios
    # Filter for failures only
    failed_df = raw_results_df[raw_results_df["result"] == 0]
    
    if not failed_df.empty:
        # Calculate counts per error type
        error_counts = (
            failed_df.groupby(["answer_generator", "suite", "final_error_type"])
            .size()
            .reset_index(name="count")
        )
        
        # Merge with total counts to calculate ratios relative to total runs
        # First, get total counts per generator/suite group
        total_counts = raw_results_df.groupby(["answer_generator", "suite"]).size().reset_index(name="total_runs")
        
        # Merge error counts with totals
        error_summary = pd.merge(error_counts, total_counts, on=["answer_generator", "suite"])
        
        # Calculate failure rate for each specific error type
        error_summary["failure_ratio"] = error_summary["count"] / error_summary["total_runs"]
        
        print(f"{bcolors.HEADER}--- Detailed Error Breakdown ---{bcolors.ENDC}")
        # Sort for better readability
        error_summary = error_summary.sort_values(["answer_generator", "suite", "count"], ascending=[True, True, False])
        print(error_summary.to_string(index=False))

        # --- DETAILED DEBUG FOR GROUND TRUTH FAILURES ---
        print(f"\n{bcolors.FAIL}--- DETAILED GROUND TRUTH FAILURES ---{bcolors.ENDC}")
        gt_failures = failed_df[failed_df["answer_generator"] == "GroundTruthAnswerGenerator"]
        if not gt_failures.empty:
            for idx, row in gt_failures.iterrows():
                print(f"\nBenchmark: {row['benchmark_name']} (Suite: {row['suite']})")
                print(f"Error Type: {row['final_error_type']}")
                print(f"Full Validation Error:\n{row['validation_error']}")
                print("-" * 60)
        else:
            print("No GroundTruth failures found in this run.")
        # -----------------------------------------------
    else:
        print(f"{bcolors.OKGREEN}No failures detected!{bcolors.ENDC}")
else:
    print("No results to analyze.")

--- Benchmark Summary ---
                                                                                                    passed  \
answer_generator                                                       suite                                 
GeminiAnswerGenerator(gemini-2.5-flash)                                api_understanding                 7   
                                                                       configure_adk_features_mc        24   
                                                                       diagnose_setup_errors_mc         20   
                                                                       fix_errors                        8   
                                                                       predict_runtime_behavior_mc       8   
GeminiAnswerGenerator(gemini-2.5-flash)-with-context-llms-relevant.txt api_understanding                28   
                                                                       configure_adk_features_

In [13]:
# --- Analysis Configuration ---
generator_to_analyze = 'GeminiAnswerGenerator(gemini-2.5-flash)' 
# result_type_to_see = 'fail'

# analyze_logs(
#     results_df=raw_results_df,
#     generator_name=generator_to_analyze,
#     result_type=result_type_to_see
# )


In [16]:
raw_results_df[(raw_results_df['suite'] == 'fix_errors')&(raw_results_df['answer_generator'] == generator_to_analyze)].final_error_type.value_counts()

final_error_type
OtherError        11
SystemExit         5
ValueError         2
AssertionError     1
Name: count, dtype: int64